In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import pygeohash as pgh

def create_advanced_features(train_path, test_path):
    print("Loading and combining data for Lag Features...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df['is_test'] = 0
    test_df['is_test'] = 1
    test_df['demand'] = np.nan

    combined = pd.concat([train_df, test_df], ignore_index=True)

    # ==========================================
    # UPGRADE 3: Advanced Temporal Lags
    # ==========================================
    # Sort chronologically to safely shift demand backward in time
    combined = combined.sort_values(['geohash', 'timestamp', 'day'])

    # Lag 1 (Yesterday's demand at this exact location & time)
    combined['demand_lag_1'] = combined.groupby(['geohash', 'timestamp'])['demand'].shift(1)

    # Lag 7 (Last week's demand at this exact location & time)
    combined['demand_lag_7'] = combined.groupby(['geohash', 'timestamp'])['demand'].shift(7)

    # Return to original order
    combined = combined.sort_index()

    print("Engineering base features...")
    combined['lat'] = combined['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else np.nan)
    combined['lon'] = combined['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else np.nan)

    def time_to_minutes(t):
        if pd.isnull(t): return np.nan
        h, m = map(int, str(t).split(':'))
        return h * 60 + m

    combined['minutes'] = combined['timestamp'].apply(time_to_minutes)
    combined['time_sin'] = np.sin(2 * np.pi * combined['minutes'] / 1440)
    combined['time_cos'] = np.cos(2 * np.pi * combined['minutes'] / 1440)

    combined['LargeVehicles'] = combined['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(-1)
    combined['Landmarks'] = combined['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(-1)
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(combined['NumberofLanes'].median())

    # Fill categoricals for CatBoost
    cat_features = ['geohash', 'RoadType', 'Weather']
    for col in cat_features:
        combined[col] = combined[col].fillna('Missing').astype(str)

    # Split back to train/test
    train_full = combined[combined['is_test'] == 0].drop(columns=['is_test', 'Index', 'timestamp', 'minutes'])
    test_full = combined[combined['is_test'] == 1].drop(columns=['is_test', 'Index', 'timestamp', 'minutes', 'demand'])

    # XGBoost needs numeric encodings for categoricals
    xgb_train = train_full.copy()
    xgb_test = test_full.copy()
    for col in cat_features:
        le = LabelEncoder()
        # Fit on combined to ensure test categories are seen
        le.fit(combined[col])
        xgb_train[col] = le.transform(xgb_train[col])
        xgb_test[col] = le.transform(xgb_test[col])

    return train_full, test_full, xgb_train, xgb_test, cat_features

if __name__ == "__main__":
    cb_train, cb_test, xgb_train, xgb_test, cat_cols = create_advanced_features('dataset/train.csv', 'dataset/test.csv')

    # ==========================================
    # UPGRADE 1: Log-Transform the Target
    # ==========================================
    # np.log1p handles 0s safely by calculating log(1 + x)
    max_day = cb_train['day'].max()
    y_full = np.log1p(cb_train['demand'])

    # Time-Series Split
    train_mask = cb_train['day'] < max_day
    val_mask = cb_train['day'] == max_day

    y_tr, y_va = y_full[train_mask], y_full[val_mask]

    print("\n--- Training CatBoost Model ---")
    X_cb_tr = cb_train[train_mask].drop(columns=['day', 'demand'])
    X_cb_va = cb_train[val_mask].drop(columns=['day', 'demand'])
    X_cb_full = cb_train.drop(columns=['day', 'demand'])
    X_cb_test = cb_test.drop(columns=['day'])

    # UPGRADE 5: Tight learning rate, high iterations
    cb_model = CatBoostRegressor(
        iterations=10000,
        learning_rate=0.01,
        depth=8,
        task_type='GPU',
        eval_metric='RMSE',
        random_seed=42
    )

    cb_model.fit(
        X_cb_tr, y_tr,
        eval_set=(X_cb_va, y_va),
        cat_features=cat_cols,
        early_stopping_rounds=200,
        verbose=500
    )

    print("\nRetraining CatBoost on 100% data...")
    cb_final = CatBoostRegressor(
        iterations=cb_model.get_best_iteration(),
        learning_rate=0.01,
        depth=8,
        task_type='GPU',
        random_seed=42
    )
    cb_final.fit(X_cb_full, y_full, cat_features=cat_cols, verbose=False)
    cb_preds_log = cb_final.predict(X_cb_test)


    print("\n--- Training XGBoost Model ---")
    X_xgb_tr = xgb_train[train_mask].drop(columns=['day', 'demand'])
    X_xgb_va = xgb_train[val_mask].drop(columns=['day', 'demand'])
    X_xgb_full = xgb_train.drop(columns=['day', 'demand'])
    X_xgb_test = xgb_test.drop(columns=['day'])

    dtrain = xgb.DMatrix(X_xgb_tr, label=y_tr)
    dval = xgb.DMatrix(X_xgb_va, label=y_va)

    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': 'cuda',
        'learning_rate': 0.01,
        'max_depth': 8,
        'min_child_weight': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8
    }

    xgb_model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=10000,
        evals=[(dval, 'Validation')],
        early_stopping_rounds=200,
        verbose_eval=500
    )

    print("\nRetraining XGBoost on 100% data...")
    dtrain_full = xgb.DMatrix(X_xgb_full, label=y_full)
    dtest = xgb.DMatrix(X_xgb_test)

    xgb_final = xgb.train(
        xgb_params,
        dtrain_full,
        num_boost_round=xgb_model.best_iteration
    )
    xgb_preds_log = xgb_final.predict(dtest)

    # ==========================================
    # UPGRADE 4: Revert Log-Transform and Ensemble
    # ==========================================
    # np.expm1 reverts the np.log1p transformation
    cb_preds = np.expm1(cb_preds_log)
    xgb_preds = np.expm1(xgb_preds_log)

    # Clip to 0 to prevent mathematically impossible negative demand predictions
    cb_preds = np.clip(cb_preds, 0, None)
    xgb_preds = np.clip(xgb_preds, 0, None)

    # Simple 50/50 Blend
    final_predictions = (cb_preds * 0.5) + (xgb_preds * 0.5)

    print("\nSaving submission...")
    test_raw = pd.read_csv('dataset/test.csv')
    sub = pd.DataFrame({
        'Index': test_raw['Index'],
        'demand': final_predictions
    })
    sub.to_csv('submission_ensemble_lag_log.csv', index=False)
    print("Done! Submission saved as 'submission_ensemble_lag_log.csv'")

Loading and combining data for Lag Features...
Engineering base features...

--- Training CatBoost Model ---
0:	learn: 0.1076314	test: 0.1121217	best: 0.1121217 (0)	total: 21.6ms	remaining: 3m 35s
500:	learn: 0.0308831	test: 0.0467122	best: 0.0458783 (395)	total: 11.7s	remaining: 3m 42s
bestTest = 0.04587832823
bestIteration = 395
Shrink model to first 396 iterations.

Retraining CatBoost on 100% data...

--- Training XGBoost Model ---
[0]	Validation-rmse:0.11216
[500]	Validation-rmse:0.05121
[1000]	Validation-rmse:0.04994
[1500]	Validation-rmse:0.04992
[1556]	Validation-rmse:0.04997

Retraining XGBoost on 100% data...

Saving submission...
Done! Submission saved as 'submission_ensemble_lag_log.csv'
